# 03 — Birleşik Cross-Entropy ve BatchNorm türevleri


In [ ]:
import random
import torch
import torch.nn.functional as F

words = open("names.txt", "r").read().splitlines()
chars = sorted(set("".join(words)))
stoi = {ch: i + 1 for i, ch in enumerate(chars)}
stoi["."] = 0
itos = {i: ch for ch, i in stoi.items()}
vocab_size = len(stoi)
block_size = 3

def build_dataset(items):
    X, Y = [], []
    for word in items:
        context = [0] * block_size
        for ch in word + ".":
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    return torch.tensor(X), torch.tensor(Y)

random.seed(42)
random.shuffle(words)
n1, n2 = int(0.8 * len(words)), int(0.9 * len(words))
Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])
print(Xtr.shape, Xdev.shape, Xte.shape, vocab_size)

In [ ]:
n_embd, n_hidden, batch_size = 10, 64, 32
n = batch_size
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((vocab_size, n_embd), generator=g)
W1 = torch.randn((n_embd * block_size, n_hidden), generator=g) * (5 / 3) / (n_embd * block_size) ** 0.5
b1 = torch.randn(n_hidden, generator=g) * 0.1
W2 = torch.randn((n_hidden, vocab_size), generator=g) * 0.1
b2 = torch.randn(vocab_size, generator=g) * 0.1
bngain = torch.randn((1, n_hidden), generator=g) * 0.1 + 1.0
bnbias = torch.randn((1, n_hidden), generator=g) * 0.1
parameters = [C, W1, b1, W2, b2, bngain, bnbias]
for p in parameters:
    p.requires_grad = True

ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
Xb, Yb = Xtr[ix], Ytr[ix]

In [ ]:
emb = C[Xb]
embcat = emb.view(emb.shape[0], -1)
hprebn = embcat @ W1 + b1

bnmeani = hprebn.sum(0, keepdim=True) / n
bndiff = hprebn - bnmeani
bndiff2 = bndiff ** 2
bnvar = bndiff2.sum(0, keepdim=True) / (n - 1)
bnvar_inv = (bnvar + 1e-5) ** -0.5
bnraw = bndiff * bnvar_inv
hpreact = bngain * bnraw + bnbias
h = torch.tanh(hpreact)
logits = h @ W2 + b2

logit_maxes = logits.max(1, keepdim=True).values
norm_logits = logits - logit_maxes
counts = norm_logits.exp()
counts_sum = counts.sum(1, keepdim=True)
counts_sum_inv = counts_sum ** -1
probs = counts * counts_sum_inv
logprobs = probs.log()
loss = -logprobs[range(n), Yb].mean()

intermediates = [
    logprobs, probs, counts, counts_sum, counts_sum_inv, norm_logits,
    logit_maxes, logits, h, hpreact, bnraw, bnvar_inv, bnvar,
    bndiff2, bndiff, hprebn, bnmeani, embcat, emb,
]
for tensor in intermediates:
    tensor.retain_grad()
for p in parameters:
    p.grad = None
loss.backward()
print(f"loss: {loss.item():.6f}")

In [ ]:
def cmp(name, manual_grad, tensor):
    exact = torch.equal(manual_grad, tensor.grad)
    approximate = torch.allclose(manual_grad, tensor.grad)
    maxdiff = (manual_grad - tensor.grad).abs().max().item()
    print(f"{name:16s} exact={str(exact):5s} approximate={str(approximate):5s} maxdiff={maxdiff:.3g}")
    assert exact or approximate

dlogprobs = torch.zeros_like(logprobs)
dlogprobs[range(n), Yb] = -1 / n
dprobs = dlogprobs / probs
dcounts_sum_inv = (counts * dprobs).sum(1, keepdim=True)
dcounts = counts_sum_inv * dprobs
dcounts_sum = -(counts_sum ** -2) * dcounts_sum_inv
dcounts += torch.ones_like(counts) * dcounts_sum
dnorm_logits = counts * dcounts
dlogits = dnorm_logits.clone()
dlogit_maxes = -dnorm_logits.sum(1, keepdim=True)
dlogits += F.one_hot(logits.max(1).indices, logits.shape[1]) * dlogit_maxes
dh = dlogits @ W2.T
dW2 = h.T @ dlogits
db2 = dlogits.sum(0)
dhpreact = (1 - h ** 2) * dh
dbngain = (bnraw * dhpreact).sum(0, keepdim=True)
dbnbias = dhpreact.sum(0, keepdim=True)
dbnraw = bngain * dhpreact
dbndiff = bnvar_inv * dbnraw
dbnvar_inv = (bndiff * dbnraw).sum(0, keepdim=True)
dbnvar = -0.5 * (bnvar + 1e-5) ** -1.5 * dbnvar_inv
dbndiff2 = torch.ones_like(bndiff2) * dbnvar / (n - 1)
dbndiff += 2 * bndiff * dbndiff2
dhprebn = dbndiff.clone()
dbnmeani = -dbndiff.sum(0, keepdim=True)
dhprebn += torch.ones_like(hprebn) * dbnmeani / n
dembcat = dhprebn @ W1.T
dW1 = embcat.T @ dhprebn
db1 = dhprebn.sum(0)
demb = dembcat.view_as(emb)
dC = torch.zeros_like(C)
dC.index_add_(0, Xb.reshape(-1), demb.reshape(-1, n_embd))

checks = [
    ("logprobs", dlogprobs, logprobs), ("probs", dprobs, probs),
    ("counts_sum_inv", dcounts_sum_inv, counts_sum_inv), ("counts_sum", dcounts_sum, counts_sum),
    ("counts", dcounts, counts), ("norm_logits", dnorm_logits, norm_logits),
    ("logit_maxes", dlogit_maxes, logit_maxes), ("logits", dlogits, logits),
    ("h", dh, h), ("W2", dW2, W2), ("b2", db2, b2),
    ("hpreact", dhpreact, hpreact), ("bngain", dbngain, bngain),
    ("bnbias", dbnbias, bnbias), ("bnraw", dbnraw, bnraw),
    ("bnvar_inv", dbnvar_inv, bnvar_inv), ("bnvar", dbnvar, bnvar),
    ("bndiff2", dbndiff2, bndiff2), ("bndiff", dbndiff, bndiff),
    ("bnmeani", dbnmeani, bnmeani), ("hprebn", dhprebn, hprebn),
    ("embcat", dembcat, embcat), ("W1", dW1, W1), ("b1", db1, b1),
    ("emb", demb, emb), ("C", dC, C),
]
for args in checks:
    cmp(*args)

In [ ]:
loss_fast = F.cross_entropy(logits, Yb)
dlogits_fast = F.softmax(logits, dim=1)
dlogits_fast[range(n), Yb] -= 1
dlogits_fast /= n

hpreact_fast = bngain * (hprebn - hprebn.mean(0, keepdim=True)) / torch.sqrt(
    hprebn.var(0, keepdim=True, unbiased=True) + 1e-5
) + bnbias
dhprebn_fast = bngain * bnvar_inv / n * (
    n * dhpreact
    - dhpreact.sum(0, keepdim=True)
    - n / (n - 1) * bnraw * (dhpreact * bnraw).sum(0, keepdim=True)
)

print("loss farkı:", (loss_fast - loss).abs().item())
print("BatchNorm ileri farkı:", (hpreact_fast - hpreact).abs().max().item())
cmp("fused logits", dlogits_fast, logits)
cmp("fused BatchNorm", dhprebn_fast, hprebn)